# 01 – Stream processing avec Spark Structured Streaming
Kafka `reviews_stream` → parsing JSON → filtres / fenêtres / statistiques → PostgreSQL.
Le producteur doit tourner (`docker compose up -d producer`).

In [1]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               BooleanType, ArrayType)
from shopstream_utils import (get_spark, read_table, write_table,
                              KAFKA_BOOTSTRAP, TOPIC, CHECKPOINT_DIR)

spark = get_spark("01-streaming", shuffle_partitions=4)

Spark 3.5.3 prêt - Spark UI : http://localhost:4041


## 2.1 Lire le topic Kafka

In [3]:
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 500)
    .load()
)

raw.printSchema()
print("Streaming :", raw.isStreaming)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

Streaming : True


## 2.2 Parser et nettoyer

In [10]:
# TODO : schéma explicite de la valeur JSON (attention à hashtags : tableau de chaînes)
schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("username", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("rating", IntegerType(), True),
    StructField("lang", StringType(), True),
    StructField("text", StringType(), True),
    StructField("hashtags", ArrayType(StringType()), True),
    StructField("helpful_votes", IntegerType(), True),
    StructField("verified_purchase", BooleanType(), True),
    StructField("country", StringType(), True)
])

# TODO : value -> string -> from_json -> colonnes à plat
#        + event_time en timestamp, text_length, kafka_partition, kafka_offset, ingest_time
#        + filtrer les lignes invalides
parsed = (
    raw
    .select(
        F.from_json(
            F.col("value").cast("string"),
            schema
        ).alias("r"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset")
    )
    .select(
        "r.*",
        "kafka_partition",
        "kafka_offset"
    )
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )
    .withColumn(
        "text_length",
        F.length("text")
    )
    .withColumn(
        "ingest_time",
        F.current_timestamp()
    )
    .filter(
        F.col("text").isNotNull()
        & F.col("event_time").isNotNull()
    )
)

parsed.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- lang: string (nullable = true)
 |-- text: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- helpful_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- text_length: integer (nullable = true)
 |-- ingest_time: timestamp (nullable = false)



In [12]:
import shutil
shutil.rmtree(f"{CHECKPOINT_DIR}/raw", ignore_errors=True)

## 2.3 Transformation 1 – filtrage (langue + mot-clé)
Aperçu avec le sink `memory` (pratique dans un notebook).

In [13]:
# TODO : avis FR uniquement + colonne booléenne mentions_delivery (livraison|delivery, insensible à la casse)
reviews_fr = (
    parsed
    .filter(F.col("lang") == "fr")
    .withColumn(
        "mentions_delivery",
        F.lower(F.col("text")).rlike("livraison|delivery")
    )
)

# Aperçu : sink memory, attendre ~20 s, interroger la table en SQL, puis ARRÊTER la requête

q_fr = (
    reviews_fr.writeStream
    .format("memory")
    .queryName("reviews_fr_mem")
    .outputMode("append")
    .start()
)

import time
time.sleep(20)

spark.sql("""
    SELECT
        review_id,
        lang,
        text,
        mentions_delivery
    FROM reviews_fr_mem
    LIMIT 20
""").show(truncate=False)
q_fr.stop()

+------------------------------------+----+------------------------------------------------------------------------------+-----------------+
|review_id                           |lang|text                                                                          |mentions_delivery|
+------------------------------------+----+------------------------------------------------------------------------------+-----------------+
|830e62d1-3d40-db6c-5685-847c3c45f24a|fr  |Emballage soigne, merci. super qualite! #jouets                               |false            |
|ac154410-4b63-1bde-c667-15fb56d539be|fr  |Parfait, rien a redire. tres satisfait de ce raquette #sport                  |false            |
|9dc0d37d-8395-b084-49ab-8a42ca7ebabf|fr  |Livraison rapide et produit conforme honnetement :( #mode #top                |true             |
|0e21d90a-10ac-d005-4532-caffd9b4a81f|fr  |Fonctionne parfaitement pour un cadeau. bouilloire correct sans plus... #happy|false            |
|71011bf6-c48

## 2.4 Sink 1 – avis bruts → PostgreSQL `reviews_stream`
Démo : `docker compose exec postgres psql -U spark -d shopstream` puis `SELECT count(*) FROM reviews_stream;` et `\watch 2`.

In [14]:
# TODO : foreachBatch -> table reviews_stream (append), trigger 10 s, checkpoint dédié
def write_raw(batch_df, batch_id):
    write_table(
        batch_df,
        "reviews_stream",
        mode="append"
    )

q_raw = (
    parsed.writeStream
    .foreachBatch(write_raw)
    .option(
        "checkpointLocation",
        f"{CHECKPOINT_DIR}/raw"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

## 2.5 Transformation 2 + Action 1 – fenêtres d'1 minute par catégorie (watermark 2 min, mode append)

In [15]:
# TODO : fenêtre fixe d'1 minute par catégorie, watermark 2 minutes, count + note moyenne
#        colonnes finales : window_start, window_end, category, nb_reviews, avg_rating

windowed = (
    parsed
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        F.window("event_time", "1 minute"),
        "category"
    )
    .agg(
        F.count("*").alias("nb_reviews"),
        F.avg("rating").alias("avg_rating")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "category",
        "nb_reviews",
        "avg_rating"
    )
)

# TODO : écriture en mode append dans stream_window_counts (foreachBatch + checkpoint dédié)
def write_window(batch_df, batch_id):
    write_table(
        batch_df,
        "stream_window_counts",
        mode="append"
    )

q_win = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(write_window)
    .option(
        "checkpointLocation",
        f"{CHECKPOINT_DIR}/window_counts"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

### Q2.2 – même agrégation en mode `update` vers la console (debug)

In [17]:
# TODO Q2.2 : même agrégation (windowed) vers le sink console en mode update, ~25 s, puis stop()
# (la sortie console apparaît dans : docker compose logs -f jupyter)
q_debug = (
    windowed.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(25)

q_debug.stop()

## 2.6 Action 2 – statistiques par hashtag (mode complete, table écrasée à chaque micro-batch)

In [18]:
# TODO : explode(hashtags) puis par hashtag : nb_reviews, avg_text_length, avg_rating
hashtag_stats = (
    parsed
    .select(
        F.explode("hashtags").alias("hashtag"),
        "text_length",
        "rating"
    )
    .groupBy("hashtag")
    .agg(
        F.count("*").alias("nb_reviews"),
        F.avg("text_length").alias("avg_text_length"),
        F.avg("rating").alias("avg_rating")
    )
)


def write_hashtag_stats(batch_df, batch_id):
    write_table(
        batch_df,
        "stream_hashtag_stats",
        mode="overwrite"
    )


q_tags = (
    hashtag_stats.writeStream
    .outputMode("complete")
    .foreachBatch(write_hashtag_stats)
    .option(
        "checkpointLocation",
        f"{CHECKPOINT_DIR}/hashtag_stats"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

## 2.7 Supervision

In [21]:
# TODO : pour chaque requête active, afficher name, status et les indicateurs de lastProgress
for q in spark.streams.active:
    print("=" * 60)
    print("ID :", q.id)
    print("Name :", q.name)
    print("Status :", q.status)

    progress = q.lastProgress

    if progress:
        print("Batch ID :", progress.get("batchId"))
        print("Nombre de lignes :", progress.get("numInputRows"))
        print("Input rate :", progress.get("inputRowsPerSecond"))
        print("Process rate :", progress.get("processedRowsPerSecond"))
        print("Durée :", progress.get("durationMs"))
    else:
        print("Pas encore de lastProgress")

ID : 20cff06b-5862-4c4c-9f03-be2b0f856533
Name : None
Status : {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID : 128
Nombre de lignes : 44
Input rate : 4.399560043995601
Process rate : 14.789915966386554
Durée : {'addBatch': 2252, 'commitOffsets': 362, 'getBatch': 0, 'latestOffset': 3, 'queryPlanning': 38, 'triggerExecution': 2975, 'walCommit': 318}
ID : d03e1e97-db7c-40d2-bf0f-a7adf8a781a5
Name : None
Status : {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID : 303
Nombre de lignes : 44
Input rate : 4.4
Process rate : 14.175257731958762
Durée : {'addBatch': 2121, 'commitOffsets': 475, 'getBatch': 0, 'latestOffset': 4, 'queryPlanning': 50, 'triggerExecution': 3104, 'walCommit': 451}
ID : acafc059-b8e1-4e61-a69c-4854a959a74a
Name : None
Status : {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID : 309
Nombre de lignes : 44
Input rate : 4.4
Process rate : 1

In [22]:
for t in ["reviews_stream", "stream_window_counts", "stream_hashtag_stats"]:
    print(t, read_table(spark, t).count())
read_table(spark, "stream_window_counts").orderBy(F.desc("window_start")).show(8)
read_table(spark, "stream_hashtag_stats").orderBy(F.desc("nb_reviews")).show(8)

reviews_stream 59453
stream_window_counts 1157
stream_hashtag_stats 27
+-------------------+-------------------+------------+----------+------------------+
|       window_start|         window_end|    category|nb_reviews|        avg_rating|
+-------------------+-------------------+------------+----------+------------------+
|2026-09-23 13:34:00|2026-09-23 13:35:00|alimentation|        54|3.8518518518518516|
|2026-09-23 13:34:00|2026-09-23 13:35:00|      jouets|        59|3.7796610169491527|
|2026-09-23 13:34:00|2026-09-23 13:35:00|      beaute|        32|             3.625|
|2026-09-23 13:34:00|2026-09-23 13:35:00|        mode|        58| 3.603448275862069|
|2026-09-23 13:34:00|2026-09-23 13:35:00|      maison|        35| 4.085714285714285|
|2026-09-23 13:34:00|2026-09-23 13:35:00|electronique|        28|3.9285714285714284|
|2026-09-23 13:34:00|2026-09-23 13:35:00|       sport|        19|3.6315789473684212|
|2026-09-23 13:34:00|2026-09-23 13:35:00|      livres|        39|3.538461538461

### Reprise sur checkpoint et contrôle des doublons

In [23]:
# TODO : arrêter q_raw, attendre, la relancer avec LE MÊME checkpoint, puis compter les doublons de review_id
before = read_table(spark, "reviews_stream").count()
print("Avant arrêt :", before)

q_raw.stop()
print("q_raw actif :", q_raw.isActive)

Avant arrêt : 63642
q_raw actif : False


In [24]:
time.sleep(15)

q_raw = (
    parsed.writeStream
    .foreachBatch(write_raw)
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/raw")
    .trigger(processingTime="10 seconds")
    .start()
)

print("q_raw relancé :", q_raw.isActive)

q_raw relancé : True


In [25]:
time.sleep(20)

after = read_table(spark, "reviews_stream").count()

print("Avant arrêt :", before)
print("Après reprise :", after)

Avant arrêt : 63642
Après reprise : 65137


In [26]:
duplicates = (
    read_table(spark, "reviews_stream")
    .groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Nombre de review_id dupliqués :", duplicates.count())

Nombre de review_id dupliqués : 0


**Laissez `q_raw` tourner** pendant les parties 3 et 4. En fin de journée :
```python
for q in spark.streams.active: q.stop()
```

## Réponses aux questions
- **Q1.1** : ...
- **Q1.2** : ...
- **Q1.3** : ...
- **Q2.1** : ...
- **Q2.2** : ...
- **Q2.3** : ...
- **Q2.4** : ...